# 04. 미래 주가 예측 (Recursive Extension)

## 📋 개요
학습된 Multi-horizon 모델을 사용하여 미래 주가를 예측합니다.

## ✨ 핵심 전략: Recursive Extension
- **Step 1**: 최신 데이터로 5일치(h1~h5) 예측
- **Step 2**: 예측값을 실제값처럼 사용하여 다음 5일 예측
- **Step 3**: 목표 기간까지 반복 확장

```
현재 데이터 → [모델] → 5일 예측값
예측값 + 현재 데이터 → [모델] → 다음 5일 예측값
예측값 + ... → [모델] → 계속 확장...
```

## 🔄 데이터 흐름
```
학습된 모델 (04_models)
    ↓
최신 Feature 데이터 (02_processed)
    ↓
미래 영업일 캘린더 (99_meta)
    ↓
[Recursive Extension Loop]
    ↓
미래 예측 결과 (03_results/forecasts)
```

## 🔧 v1.1 패치 노트 (2026-02-08)
- **수정**: Chunk 데이터 오염 방지
  - 예측값 추가 시 `volume` 등 실제 과거 데이터 참조 제거
  - 최근 평균값 또는 0으로 대체하여 예측 순수성 유지

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
from datetime import datetime, timedelta

from src.utils.config import load_config, ProjectPaths
from src.models.lightgbm_model import LightGBMModel
from src.features.technical import (
    calc_sma, calc_rsi, calc_macd, calc_bollinger, calc_volume_ratio
)

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로

In [ ]:
# ==========================================
# 1. 설정 로드
# ==========================================
cfg = load_config()
paths = ProjectPaths.from_config(cfg)

# 경로 자동 생성
paths.ensure_dirs()

# ==========================================
# 2. 경로 설정
# ==========================================
meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))

## 2️⃣ 파일 존재 확인

In [ ]:
# ==========================================
# 필수 파일 존재 여부 확인
# ==========================================

# 1. 학습된 모델 파일
model_path = paths.get_model_path("lightgbm_multi")
print(f"✅ 모델 파일 확인: {model_path.name}")

# 2. Feature 데이터셋
dataset_path = paths.get_dataset_parquet()
if not dataset_path.exists():
    raise FileNotFoundError(f"❌ 데이터셋을 찾을 수 없습니다: {dataset_path}")
print(f"✅ 데이터셋 확인: {dataset_path.name}")

# 3. 영업일 캘린더
calendar_path = meta_dir / 'krx_calendar.csv'
if not calendar_path.exists():
    raise FileNotFoundError(
        f"❌ 영업일 캘린더를 찾을 수 없습니다: {calendar_path}\n"
        "💡 Tip: 99_save_trading_days.ipynb를 먼저 실행하세요."
    )
print(f"✅ 영업일 캘린더 확인: {calendar_path.name}")

## 3️⃣ 데이터 로드

In [ ]:
# ==========================================
# 1. 학습된 모델 로드
# ==========================================
print("\n📦 모델 로드 중...")
model = LightGBMModel.load(str(model_path))

print(f"   - 모델명: {model.model_name}")
print(f"   - 버전: {model.model_version}")
print(f"   - Horizons: {model.target_columns}")
print(f"   - Feature 개수: {len(model.feature_list)}")

In [ ]:
# ==========================================
# 2. Feature 데이터셋 로드
# ==========================================
print("\n📥 Feature 데이터 로드 중...")
df_features = pd.read_parquet(dataset_path)
df_features['date'] = pd.to_datetime(df_features['date'])
df_features = df_features.sort_values(['ticker', 'date']).reset_index(drop=True)

# 최신 날짜 확인
last_date = df_features['date'].max()
print(f"   - 총 행수: {len(df_features):,}")
print(f"   - 종목 수: {df_features['ticker'].nunique()}")
print(f"   - 최신 날짜: {last_date.strftime('%Y-%m-%d')}")

In [ ]:
# ==========================================
# 3. 영업일 캘린더 로드 및 예측 기간 설정
# ==========================================
print("\n📅 예측 기간 설정 중...")
df_calendar = pd.read_csv(calendar_path)
df_calendar['date'] = pd.to_datetime(df_calendar['date'])

# 최신 날짜 이후의 영업일만 추출
future_dates = df_calendar[df_calendar['date'] > last_date]['date'].values

if len(future_dates) == 0:
    raise ValueError(
        f"❌ 예측할 미래 영업일이 없습니다.\n"
        f"   최신 데이터: {last_date.strftime('%Y-%m-%d')}\n"
        f"   캘린더 종료일: {df_calendar['date'].max().strftime('%Y-%m-%d')}\n"
        "💡 Tip: 99_save_trading_days.ipynb에서 기간을 연장하세요."
    )

print(f"   - 예측 시작일: {pd.to_datetime(future_dates[0]).strftime('%Y-%m-%d')}")
print(f"   - 예측 종료일: {pd.to_datetime(future_dates[-1]).strftime('%Y-%m-%d')}")
print(f"   - 총 예측일수: {len(future_dates)}일")

## 4️⃣ Feature 생성 함수 정의

In [ ]:
def calculate_features_for_ticker(df_ticker: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    단일 종목에 대한 기술적 지표 계산
    (src/features/builder.py의 로직 재활용)
    """
    df = df_ticker.copy()
    params = config['preprocessing']
    
    # 1. 이동평균 (MA)
    for window in params['technical_windows']:
        df[f'feature_ma_{window}'] = calc_sma(df['close'], window)
    
    # 2. 변동성 (20일 표준편차)
    df['feature_volatility_20'] = df['close'].pct_change().rolling(20).std()
    
    # 3. 거래량 비율
    df['feature_volume_ratio'] = calc_volume_ratio(df['volume'], params['volume_window'])
    
    # 4. RSI
    df['feature_rsi_14'] = calc_rsi(df['close'], params['rsi_period'])
    
    # 5. MACD
    macd, signal, hist = calc_macd(df['close'])
    df['feature_macd'] = macd
    df['feature_macd_signal'] = signal
    df['feature_macd_hist'] = hist
    
    # 6. Bollinger Bands
    upper, mid, lower = calc_bollinger(df['close'])
    df['feature_bb_upper'] = upper
    df['feature_bb_middle'] = mid
    df['feature_bb_lower'] = lower
    
    # 7. Meta Features (간단 버전)
    df['liquidity_score'] = (df['close'] * df['volume']).rolling(20).mean()
    df['risk_composite'] = df['feature_volatility_20'].fillna(0)
    
    return df

print("✅ Feature 계산 함수 정의 완료")

## 5️⃣ Recursive Extension 예측 실행

### 🔧 v1.1 패치: 데이터 오염 방지

**문제**: 
- 예측값을 추가할 때 `latest_row.get('volume', 0)` 처럼 실제 과거 데이터를 참조하면
- Chunk 1 이후 예측값과 실제값이 혼합되어 Feature 재계산 시 오염 발생

**해결**:
- `volume`: 최근 20일 평균 사용 (예측 불가능한 값이므로)
- 또는 0으로 설정 (volume 기반 피처가 적다면)

In [ ]:
# ==========================================
# Recursive Extension 파라미터
# ==========================================
CHUNK_SIZE = 5  # h1~h5 모델이 한 번에 예측하는 일수
NUM_CHUNKS = int(np.ceil(len(future_dates) / CHUNK_SIZE))

print(f"\n🔄 Recursive Extension 시작")
print(f"   - Chunk 크기: {CHUNK_SIZE}일")
print(f"   - 총 Chunk 수: {NUM_CHUNKS}개")
print(f"   - 예측 총 일수: {len(future_dates)}일\n")

# ==========================================
# 종목별 예측 루프
# ==========================================
all_forecasts = []
tickers = df_features['ticker'].unique()

for ticker in tqdm(tickers, desc="예측 진행 중"):
    # 1. 종목 데이터 추출
    df_ticker = df_features[df_features['ticker'] == ticker].copy()
    df_ticker = df_ticker.sort_values('date').reset_index(drop=True)
    
    # 🔧 v1.1: 최근 거래량 평균 사전 계산 (데이터 오염 방지)
    recent_volume_mean = df_ticker['volume'].tail(20).mean()
    
    # 예측 결과 저장용 리스트
    forecast_rows = []
    
    # 2. Chunk 단위 반복 예측
    for chunk_idx in range(NUM_CHUNKS):
        # 현재 Chunk의 예측 날짜들
        chunk_start = chunk_idx * CHUNK_SIZE
        chunk_end = min(chunk_start + CHUNK_SIZE, len(future_dates))
        chunk_dates = future_dates[chunk_start:chunk_end]
        
        # Feature 재계산 (최신 데이터 반영)
        df_ticker = calculate_features_for_ticker(df_ticker, cfg)
        
        # 최신 행 추출 (Feature 예측에 사용)
        latest_row = df_ticker.iloc[-1]
        
        # 3. 각 Horizon별 예측 (h1~h5)
        for h_idx, target_name in enumerate(model.target_columns, 1):
            if h_idx > len(chunk_dates):
                break  # 마지막 Chunk가 5일 미만인 경우
            
            # Feature DataFrame 준비
            X_pred = pd.DataFrame([latest_row[model.feature_list].values],
                                  columns=model.feature_list)
            
            # 예측 수행
            pred_log_close = model.predict(X_pred, target_name=target_name).iloc[0]
            pred_close = np.exp(pred_log_close)
            
            # 예측 날짜
            pred_date = pd.to_datetime(chunk_dates[h_idx - 1])
            
            # 4. ✅ 예측값을 데이터에 추가 (다음 예측에 사용)
            # 🔧 v1.1 패치: 실제 과거 데이터 참조 제거
            new_row = {
                'date': pred_date,
                'ticker': ticker,
                'close': pred_close,
                'open': pred_close,  # 예측값으로 대체
                'high': pred_close * 1.02,  # 단순 추정
                'low': pred_close * 0.98,
                'volume': recent_volume_mean,  # ✅ 최근 평균 사용 (오염 방지)
                'target_log_close': pred_log_close
            }
            
            df_ticker = pd.concat([
                df_ticker,
                pd.DataFrame([new_row])
            ], ignore_index=True)
            
            # 결과 저장
            forecast_rows.append({
                'date': pred_date,
                'ticker': ticker,
                'horizon': h_idx,
                'chunk_idx': chunk_idx,
                'pred_log_close': pred_log_close,
                'pred_close': pred_close
            })
    
    # 종목별 예측 완료
    all_forecasts.extend(forecast_rows)

print("\n✅ Recursive Extension 완료")

## 6️⃣ 예측 결과 저장

In [ ]:
# ==========================================
# 1. DataFrame 변환 및 정렬
# ==========================================
df_forecasts = pd.DataFrame(all_forecasts)
df_forecasts = df_forecasts.sort_values(['ticker', 'date']).reset_index(drop=True)

# ==========================================
# 2. 통합 Parquet 저장
# ==========================================
forecast_parquet_path = paths.get_forecasts_parquet()
df_forecasts.to_parquet(forecast_parquet_path, index=False)
print(f"\n💾 예측 결과 저장 완료")
print(f"   - 파일: {forecast_parquet_path}")
print(f"   - 총 예측 행수: {len(df_forecasts):,}")

# ==========================================
# 3. 샘플 확인
# ==========================================
print("\n📊 예측 결과 샘플 (처음 10행):")
display(df_forecasts.head(10))

In [ ]:
# ==========================================
# 4. 종목별 개별 CSV 저장 (옵션)
# ==========================================
SAVE_INDIVIDUAL_CSV = cfg['preprocessing'].get('save_csv', False)

if SAVE_INDIVIDUAL_CSV:
    csv_dir = paths.get_forecasts_csv_dir()
    csv_dir.mkdir(exist_ok=True)
    
    print(f"\n📁 종목별 CSV 저장 중... ({csv_dir})")
    
    # ticker_name_map 로드 (01단계 master 활용)
    try:
        master_path = paths.get_ticker_master()
        df_master = pd.read_csv(master_path)
        ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
        print(f"   - ticker_master 로드 완료: {len(ticker_name_map)}개 종목")
    except Exception as e:
        print(f"   ⚠️  ticker_master 로드 실패, 종목코드로 저장: {e}")
        ticker_name_map = {}
    
    for ticker, group in tqdm(df_forecasts.groupby('ticker'), desc="CSV 저장"):
        # 종목명 또는 코드 사용
        name = ticker_name_map.get(str(ticker), f"ticker_{ticker}")
        safe_name = str(name).replace('/', '_').replace('\\', '_')
        group.to_csv(csv_dir / f"{safe_name}_forecast.csv", 
                     index=False, encoding='utf-8-sig')
    
    print(f"   ✅ 총 {df_forecasts['ticker'].nunique()}개 CSV 파일 저장 완료")
else:
    print("\n⏭️  개별 CSV 저장 건너뜀 (config.yaml에서 save_csv=True로 변경 가능)")

## 7️⃣ 예측 요약 통계

In [ ]:
print("\n" + "="*65)
print("📈 예측 결과 요약")
print("="*65)

# 기본 통계
print(f"\n[기본 정보]")
print(f"   - 총 종목 수: {df_forecasts['ticker'].nunique():,}개")
print(f"   - 예측 기간: {df_forecasts['date'].min().strftime('%Y-%m-%d')} ~ "
      f"{df_forecasts['date'].max().strftime('%Y-%m-%d')}")
print(f"   - 총 예측 건수: {len(df_forecasts):,}건")

# 예측 가격 분포
print(f"\n[예측 가격 통계]")
print(f"   - 평균 예측가: {df_forecasts['pred_close'].mean():,.0f}원")
print(f"   - 중앙값: {df_forecasts['pred_close'].median():,.0f}원")
print(f"   - 최소값: {df_forecasts['pred_close'].min():,.0f}원")
print(f"   - 최대값: {df_forecasts['pred_close'].max():,.0f}원")

# Chunk별 분포
print(f"\n[Chunk별 예측 건수]")
chunk_counts = df_forecasts.groupby('chunk_idx').size()
for chunk_idx, count in chunk_counts.items():
    print(f"   - Chunk {chunk_idx}: {count:,}건")

print("\n" + "="*65)
print("✅ [Step 4] 미래 주가 예측 완료")
print("="*65)
print(f"\n💡 다음 단계: 05단계에서 Universe 필터링 및 전략 백테스트 수행")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
- **통합 예측 결과**: `data/03_results/{ref_date}/forecasts/future_forecasts.parquet`
- **개별 CSV** (옵션): `data/03_results/{ref_date}/forecasts/csv/{ticker}_forecast.csv`

### 📊 예측 결과 구조
| 컬럼명 | 설명 |
|--------|------|
| `date` | 예측 대상 날짜 |
| `ticker` | 종목 코드 |
| `horizon` | 예측 시차 (1~5) |
| `chunk_idx` | Chunk 번호 (Recursive Extension 단계) |
| `pred_log_close` | 예측 로그 종가 |
| `pred_close` | 예측 종가 (원화) |

### 🔧 v1.1 패치 적용 완료
- **수정 내용**: Recursive Extension 시 데이터 오염 방지
- **변경 사항**: `volume` 값을 실제 과거 데이터 참조 대신 최근 평균 사용
- **효과**: Chunk 1+ 예측 품질 향상, Feature 재계산 시 순수성 유지

### 🚀 다음 작업
1. **Universe 필터링** (05단계):
   - `liquidity_score`, `risk_composite` 기준 적용
   - `is_suspended`, `is_delisted` 제외
   
2. **전략 백테스트**:
   - 예측 신뢰도 검증
   - 포트폴리오 구성
   - 리스크 관리

3. **모니터링**:
   - 실제 발생값과 예측값 비교
   - 모델 성능 추적

---

**Last Updated**: 2026-02-08 (v1.1 패치)  
**Pipeline Step**: 04 (Future Forecasting)  
**Method**: Recursive Extension (5-day chunks)